In [5]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam

In [6]:
# Defining the model with hyperparameters

def build_model(hp):
  model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
    Dense(10, activation='softmax')
  ])

  model.compile(
    optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
  )
  
  return model

In [7]:
# Configuring the search

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

In [8]:
# Running the hyperparameter search

(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0
tuner.search(x_train, y_train, epochs=5,
validation_data=(x_val,y_val))

Trial 10 Complete [00h 01m 17s]
val_accuracy: 0.978549987077713

Best val_accuracy So Far: 0.9796499907970428
Total elapsed time: 00h 10m 05s


In [10]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal number of units in the first dense layer is {best_hps.get('units')}.
The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}
""")

model = tuner.hypermodel.build(best_hps)
model.summary()


The optimal number of units in the first dense layer is 416.
The optimal learning rate for the optimizer is 0.0006310405988733197



c:\Users\anima\AppData\Local\pypoetry\Cache\virtualenvs\imb-ai-engineering-KgAhWUw3-py3.10\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 416)            │       326,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         4,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 330,730 (1.26 MB)

 Trainable params: 330,730 (1.26 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Training the optimized model

model.fit(x_train, y_train, epochs=10, validation_split=0.2)
test_loss, test_acc = model.evaluate(x_val, y_val)
print(f'Test accuracy: {test_acc}')

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8751 - loss: 0.4440 - val_accuracy: 0.9597 - val_loss: 0.1393
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9640 - loss: 0.1199 - val_accuracy: 0.9699 - val_loss: 0.0987
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9792 - loss: 0.0707 - val_accuracy: 0.9736 - val_loss: 0.0867
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9867 - loss: 0.0470 - val_accuracy: 0.9756 - val_loss: 0.0803
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9901 - loss: 0.0342 - val_accuracy: 0.9771 - val_loss: 0.0773
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9926 - loss: 0.0254 - val_accuracy: 0.9739 - val_loss: 0.0869
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9957 - loss: 0.0176 - val_accuracy: 0.9790 - val_loss: 0.0768
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9961 - loss: 0.0152 - 